# Fresh vs Rotten Fruit Classification — Baseline CNN

This notebook develops and trains a baseline Convolutional Neural Network (CNN)
for six-class fresh-versus-rotten fruit classification.

## Objective

Build a baseline CNN that takes a preprocessed fruit image of size
224 × 224 × 3 and predicts one of the six fruit classes.

## Classes

- Fresh Apple
- Fresh Banana
- Fresh Orange
- Rotten Apple
- Rotten Banana
- Rotten Orange

## 1. Baseline CNN Development

The baseline CNN will provide a reference model for the project.

The model will be trained using the previously verified 70% training,
15% validation, and 15% testing split.

Training will be performed in Google Colab using the available GPU.

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Baseline CNN Architecture

The baseline model is designed as a simple convolutional neural network
trained from scratch.

### Architecture

Input: 224 × 224 × 3

1. Convolutional layer — 32 filters
2. ReLU activation
3. Max pooling
4. Convolutional layer — 64 filters
5. ReLU activation
6. Max pooling
7. Convolutional layer — 128 filters
8. ReLU activation
9. Max pooling
10. Global Average Pooling
11. Dense layer — 128 units
12. Dropout
13. Output layer — 6 units with softmax

The model will serve as the baseline CNN for the six-class fruit
freshness classification task.

## 3. Build the Baseline CNN

The baseline CNN is trained from scratch using the preprocessed
224 × 224 × 3 fruit images.

The network uses three convolutional blocks followed by global
average pooling and fully connected classification layers.

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),

    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(6, activation="softmax")
])

model.summary()

## 4. Compile the Baseline CNN

The CNN is compiled using:

- Sparse categorical cross-entropy as the loss function
- Adam as the optimizer
- Accuracy as the evaluation metric

Integer class labels are used because the dataset labels are encoded as
class indices from 0 to 5.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

print("Model compiled successfully.")

## 5. Baseline CNN Forward-Pass Test

Before training, the model is tested with a single input tensor to verify
that the input shape and six-class output are correct.

In [ ]:
import numpy as np

sample_input = np.random.random((1, 224, 224, 3)).astype(np.float32)

prediction = model.predict(sample_input, verbose=0)

print("Input shape:", sample_input.shape)
print("Output shape:", prediction.shape)
print("Output probabilities:", prediction)
print("Probability sum:", prediction.sum())

## 6. Connect the Real Dataset

The baseline CNN will use the same reproducible 70% training,
15% validation, and 15% testing split created during preprocessing.

The split metadata is loaded from the project's CSV files, while the
actual images are accessed from the Kaggle dataset in Google Colab.

In [ ]:
import kagglehub
import pandas as pd
from pathlib import Path
import tensorflow as tf
import numpy as np

# Get the dataset in the Colab runtime.
dataset_path = kagglehub.dataset_download(
    "sriramr/fruits-fresh-and-rotten-for-classification"
)

DATASET_ROOT = Path(dataset_path) / "dataset"

# Load the exact split metadata stored in our GitHub repository.
GITHUB_RAW = (
    "https://raw.githubusercontent.com/"
    "onkar-726/fresh-rotten-fruit-classification/main/data"
)

train_df = pd.read_csv(f"{GITHUB_RAW}/train.csv")
validation_df = pd.read_csv(f"{GITHUB_RAW}/validation.csv")
test_df = pd.read_csv(f"{GITHUB_RAW}/test.csv")

print("Training rows  :", len(train_df))
print("Validation rows:", len(validation_df))
print("Testing rows   :", len(test_df))
print("Total rows     :", len(train_df) + len(validation_df) + len(test_df))

## 7. Create TensorFlow Datasets

In [ ]:
CLASS_NAMES = [
    "freshapples",
    "freshbanana",
    "freshoranges",
    "rottenapples",
    "rottenbanana",
    "rottenoranges",
]

CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32


def load_and_preprocess_image(image_path, label):
    image_bytes = tf.io.read_file(image_path)

    image = tf.image.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0

    return image, label


def create_dataset(dataframe, shuffle=False):
    image_paths = [
        str(DATASET_ROOT / image_path)
        for image_path in dataframe["image_path"]
    ]

    labels = (
        dataframe["class_name"]
        .map(CLASS_TO_INDEX)
        .to_numpy(dtype=np.int32)
    )

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=42,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_and_preprocess_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


train_ds = create_dataset(train_df, shuffle=True)
validation_ds = create_dataset(validation_df, shuffle=False)
test_ds = create_dataset(test_df, shuffle=False)

print("TensorFlow datasets created successfully.")

In [ ]:
for images, labels in train_ds.take(1):
    print("Batch image shape :", images.shape)
    print("Batch label shape :", labels.shape)
    print("Image dtype       :", images.dtype)
    print("Minimum pixel     :", tf.reduce_min(images).numpy())
    print("Maximum pixel     :", tf.reduce_max(images).numpy())

    predictions = model(images, training=False)

    print("CNN output shape  :", predictions.shape)